# Act 1 — Cloud SQL: managed open-source

Cloud SQL is the managed-database product for the three engines most production workloads already use: **PostgreSQL**, **MySQL**, and **SQL Server**. You don't pick a custom GCP query engine; you pick the OSS engine you'd run anyway and let GCP handle the operating system, patching, backups, replication, and the long tail of database operations.

The trade-off is exactly the one PaaS asks of you in notebook 04: you lose root on the VM and gain freedom from operating it. For 90% of production OLTP workloads, that's the right trade. The remaining 10% — extreme scale, multi-region writes, very specific tuning — show up later in the chapter (AlloyDB) or in notebook 09 (Spanner).

## Three engines, one managed surface

| Engine | Versions | When to pick |
|---|---|---|
| **PostgreSQL** | 13–17 | The modern default. Strong feature set, well-tooled, good extension surface (Cloud SQL supports a curated extension list). |
| **MySQL** | 5.7, 8.0, 8.4 | Pick when you have existing MySQL workloads or strong MySQL preference. Less ambitious than PG for new work in 2026. |
| **SQL Server** | 2017, 2019, 2022 — Standard / Enterprise / Express / Web editions | Pick when licensing or app-server compatibility forces it (BYOL Windows app stacks, Microsoft-shop migrations). |

**Machine shapes** mirror Compute Engine's general-purpose families. You pick vCPU and memory, and Cloud SQL provisions a VM behind the scenes. Sizing rule: under-provision on CPU, over-provision on memory. Postgres and MySQL both benefit far more from RAM headroom than from extra cores.

## High availability — regional persistent disk + failover

Cloud SQL's HA shape is consistent across engines:

- **Primary instance** in one zone of a region.
- **Standby** in a second zone, kept in sync via **regional Persistent Disk** (synchronous block-level replication — notebook 05).
- On primary failure, Cloud SQL **fails over** to the standby in seconds, keeping the same connection endpoint.

This is a zone-failure HA story, not a region-failure one. For cross-region disaster recovery you use **cross-region read replicas** that can be promoted to primary, with a manual promotion step (notebook 13). For active-active multi-region writes you need Spanner or AlloyDB Omni — Cloud SQL doesn't do that.

**Read replicas** are async followers in the same region or another region. They handle read scaling for read-heavy workloads (BI dashboards, search-prefetch). They lag the primary by seconds normally, more under load — never use a read replica for a query whose answer must reflect a write that just happened.

## Connecting to Cloud SQL

Three ways to reach a Cloud SQL instance:

- **Public IP + authorized networks** — the legacy default. Cloud SQL has a public endpoint; you allowlist source CIDRs. Acceptable for development, never the right answer in production.
- **Private IP via Private Service Access (legacy)** — Cloud SQL lives in a Google-owned VPC peered to yours over an allocated CIDR. Most current production deployments still use this.
- **Private Service Connect (modern)** — Cloud SQL publishes a PSC endpoint that you reach via an IP in your own VPC. No CIDR allocation, no peering. The right default for new instances.

**Cloud SQL Auth Proxy** is the canonical Cloud SQL client. It runs as a sidecar (Cloud Run, GKE) or a process (Compute Engine) and:

- Authenticates to Cloud SQL using the workload's IAM identity (Workload Identity from notebook 02, attached SA from notebook 04).
- Establishes the TLS tunnel to the instance.
- Exposes a local TCP socket your app connects to as `127.0.0.1:5432`.

With the Auth Proxy you stop managing DB connection passwords for service accounts — the SA itself authenticates, gated by `roles/cloudsql.client`. Combine with **IAM database authentication** to use the same SA for the in-database user, eliminating per-user passwords entirely.

## Backups and maintenance

**Automated backups** run daily on a window you pick; they're stored in a Google-managed location and used for point-in-time-recovery (PITR) with a 7-day default retention (configurable up to 365 days). PITR replays binary logs (MySQL) or WAL (Postgres) to restore to any second in the retention window.

**Maintenance windows** are when Cloud SQL applies engine and OS patches. Pick a low-traffic window per instance. On HA instances, maintenance is *applied to the standby first, then a failover, then the primary* — usually a few seconds of unavailability.

The boring operational ones: connection limits, IAM database auth bindings, the `cloudsql.iam_authentication` flag for Postgres. None of this is glamorous, all of it matters when a real incident hits.

# Act 2 — AlloyDB: Postgres at GCP scale

AlloyDB for PostgreSQL is a Postgres-compatible database engine that Google built on top of GCP's storage layer. It's wire-compatible with Postgres (your app drivers don't change), but the storage architecture and query engine are different — designed for OLTP-plus-analytics workloads at scale that would stress Cloud SQL Postgres.

## What makes AlloyDB different

- **Disaggregated storage.** Compute (the Postgres-compatible engine) and storage (a multi-zone distributed log + page service) are separated. You scale compute and storage independently; failover is fast because storage isn't tied to a VM.
- **Columnar engine.** AlloyDB has an in-memory columnar accelerator for analytical queries. The same database serves OLTP (row-oriented) and OLAP (column-oriented) workloads — HTAP in marketing speak — without you running ETL to a separate analytics store.
- **Cross-region replicas.** Async replicas in other regions for DR or geo-distributed reads.
- **Machine learning integration.** Native model-inference functions in SQL, so you can call Vertex AI models from a query.

**Positioning vs Cloud SQL Postgres:**

- Cloud SQL Postgres for: standard OLTP, sub-10TB databases, normal scale, normal cost.
- AlloyDB for: large Postgres workloads (TBs+), analytics-on-OLTP without ETL, very high read throughput.

AlloyDB is more expensive than Cloud SQL Postgres at small sizes — it's optimised for the scale where Cloud SQL starts to strain.

**AlloyDB Omni** is the same engine packaged to run anywhere — your data center, AWS, Azure, on the developer's laptop in Docker. Used in hybrid architectures and for self-managed deployments where you want the AlloyDB engine without the GCP managed service.

## Spanner — out of scope here, in scope in notebook 09

Spanner is GCP's globally distributed strong-consistency SQL database. It belongs in this chapter conceptually, but its shape — distributed storage, TrueTime, multi-region configurations — makes more sense alongside the other GCP-native scale databases (Bigtable, BigQuery) in notebook 09.

The one-liner for now: **use Spanner when you need horizontal write scale across regions with ACID transactions, and only then.** It's the most powerful and most expensive option in this space.

# Act 3 — Memorystore: managed in-memory

When your relational database becomes the bottleneck, the next move is rarely "buy more database" — it's "cache in front of the database." Memorystore is GCP's managed in-memory store, available in three flavours.

## Three engines

| Engine | Strengths | Use when |
|---|---|---|
| **Redis** | Rich data types (lists, hashes, sorted sets, streams), Lua, transactions, pub/sub | Default — most workloads want Redis features |
| **Memcached** | Simple, multi-threaded, large-instance-fast | Pure GET/SET caching at scale, no need for data types |
| **Valkey** | Open-source Redis fork after the Redis license change | New deployments wanting Redis API + open-source guarantees |

**Memorystore for Redis** has tiers:

- **Basic** — single node, no replication. For dev/test or non-critical caches.
- **Standard** — primary + replica in different zones, automatic failover. Production default.
- **Memorystore for Redis Cluster** — sharded, multi-node, horizontal scale. For very large caches that don't fit one node.

**Networking.** Memorystore lives in a Google-managed VPC accessed via Private Service Access (legacy) or PSC (modern). Same connectivity model as Cloud SQL.

Return to caching strategies next.

## Cache patterns

**Lazy loading (cache-aside).** Application code checks the cache first; on miss, reads the database and writes the result back to the cache. Pros: cache stays cold for never-read data. Cons: every miss is a database hit, which can stampede during traffic spikes.

**Write-through.** Every database write also writes the corresponding cache entry. Pros: cache is always fresh. Cons: more write traffic to the cache, cache pollution for data that's rarely read.

**Write-behind (write-back).** Writes go to the cache first, asynchronously flushed to the database. Pros: extremely fast writes. Cons: cache failure can lose data — only safe with replicated caches and ability to re-derive lost writes.

**TTL-based expiry.** Every cache entry has a time-to-live; expired entries are re-fetched. Pros: bounded staleness without explicit invalidation. Cons: stale data within the TTL window.

**Common patterns by use case:**

- **Sessions** — lazy load with TTL matching session length. Redis hashes are the canonical store.
- **Leaderboards** — Redis sorted sets, computed in cache only (the database doesn't need to know).
- **Hot reads** — lazy load with TTL ~60s on highly-queried records (homepage, product detail).
- **Counters** — `INCR` in Redis, periodically flushed to the database via a background job.

The cache is not the database. Anything that *only* exists in cache must be recomputable from the database — otherwise an outage loses data.

# Act 4 — Choose-what

A short decision tree to close out. The hard questions are mostly about transactional shape and scale.

## Decision tree — relational on GCP

1. **Standard OLTP, single-region, under a few TB?** → Cloud SQL (Postgres if you're picking fresh; MySQL if existing; SQL Server if licensing forces it).
2. **Postgres-shaped, but tens of TB or HTAP-style workload?** → AlloyDB for PostgreSQL.
3. **Globally distributed writes, strong consistency, ACID?** → Spanner (notebook 09).
4. **Anything-key-value, schemaless, document-oriented, real-time, or scale beyond SQL?** → notebook 09 (Firestore, Bigtable, BigQuery).
5. **Add caching when database hits are repeated, not as the first move.** → Memorystore.

**Three anti-patterns to avoid:**

- "We need Spanner because we want global redundancy." — Cloud SQL with cross-region replicas covers most DR. Spanner is for horizontal *write* scale; it's overkill (and expensive) for HA alone.
- "We'll start on Cloud SQL and migrate to Spanner later." — Spanner's schema model and lack of certain Postgres features (extensions, triggers, certain index types) make migration non-trivial. Pick deliberately at design time.
- "Memcached is faster than Redis." — Memcached has been faster for pure-string GET/SET at very high QPS historically. For most workloads, Redis's data types and persistence options earn their cost; pick Memcached only when you've measured and need it.

## What carries into later chapters

Cloud SQL and AlloyDB sit behind Cloud Run services (notebook 04) via the Cloud SQL Auth Proxy or PSC. Memorystore sits in the same VPC and is reached via private IP. Notebook 09 picks up NoSQL and analytics, and goes deep on Spanner (the database deferred from this chapter). Notebook 13 covers backup, point-in-time recovery, and cross-region DR for these stores.

Three habits to carry forward:

- **Private IP / PSC over public IP for any production database.** Public-IP databases are a security regression.
- **Workload Identity + IAM database authentication.** No DB passwords in app config.
- **Cache aggressively, but never as the only copy of state.** A cache outage should degrade performance, not lose data.